# 中证800 V47 Anchor 增强实验

目标：不再改 label，不再使用 early stopping，在 V46 强 anchor 上做最小可归因增强。

固定项：

- label：`alpha_1m`
- 训练窗口：按日期配置，默认 `2019-01-01` 到 `2025-03-31`
- 训练方式：全量训练，固定轮数，无 early stopping
- pkl 协议：兼容现有 `jq_backtest_v46_legacy_unsealed.py`

本轮只测试两个变量：

1. 参数 family：baseline / balanced / strong_reg / smooth
2. sample weight：flat / top_tail_v1

输出：8 个可直接上传聚宽回测的 pkl，以及一个 manifest。

In [ ]:
import os
import gc
import pickle
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

DATA_PATH = "train_csi800_factor_v40_data_enhancement.csv"
OUT_DIR = "csi800_ml_v47_anchor_enhancement_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_START = "2019-01-01"
TRAIN_END = "2025-03-31"
LABEL_END = "2025-03-31"
REQUIRE_LABEL_END_WITHIN_TRAIN = False  # 保持 V46 legacy_unsealed 口径

BENCHMARK = "000906.XSHG"
TARGET_COL = "alpha_1m"
TOP_N_CANDIDATES = 30
STOCK_NUM = 10
INDUSTRY_CAP_RATIO = 0.20
CORR_THRESHOLD = 0.70
DIAG_VALID_FRAC = 0.20
DIAG_VALID_MIN_MONTHS = 6
SEED = 42

print("DATA_PATH =", DATA_PATH)
print("OUT_DIR =", OUT_DIR)
print("target =", TARGET_COL)


In [ ]:
BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "roe_ttm", "roa_ttm",
    "gross_profit_ttm", "operating_profit_to_total_profit", "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage", "adjusted_profit_to_total_profit", "ACCA", "growth",
    "net_working_capital", "operating_profit_per_share", "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share", "super_quick_ratio", "MLEV", "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio", "momentum", "Rank1M", "sharpe_ratio_60", "Variance20",
    "liquidity", "beta", "ATR6", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    "Skewness20", "Kurtosis20",
]

HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60",
    "liq_paused_count_20",
    "px_close_to_ma60",
    "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m",
    "ts_Rank1M_rank_chg_1m",
]

CANDIDATE_COLS = BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS

PARAM_SPECS = [
    {
        "param_name": "v46_base_ff10",
        "fixed_iter": 120,
        "params": {
            "objective": "regression", "metric": "l2", "boosting_type": "gbdt",
            "learning_rate": 0.05, "num_leaves": 31, "min_data_in_leaf": 200,
            "feature_fraction": 1.0, "bagging_fraction": 0.8, "bagging_freq": 1,
            "lambda_l1": 0.1, "lambda_l2": 0.3, "verbose": -1,
        },
    },
    {
        "param_name": "balanced_reg",
        "fixed_iter": 160,
        "params": {
            "objective": "regression", "metric": "l2", "boosting_type": "gbdt",
            "learning_rate": 0.035, "num_leaves": 21, "max_depth": 5, "min_data_in_leaf": 300,
            "feature_fraction": 0.80, "bagging_fraction": 0.80, "bagging_freq": 1,
            "lambda_l1": 0.5, "lambda_l2": 2.0, "min_gain_to_split": 0.00001, "verbose": -1,
        },
    },
    {
        "param_name": "strong_reg",
        "fixed_iter": 180,
        "params": {
            "objective": "regression", "metric": "l2", "boosting_type": "gbdt",
            "learning_rate": 0.03, "num_leaves": 21, "max_depth": 4, "min_data_in_leaf": 500,
            "feature_fraction": 0.70, "bagging_fraction": 0.75, "bagging_freq": 1,
            "lambda_l1": 1.0, "lambda_l2": 5.0, "min_gain_to_split": 0.00001, "verbose": -1,
        },
    },
    {
        "param_name": "smooth_reg",
        "fixed_iter": 240,
        "params": {
            "objective": "regression", "metric": "l2", "boosting_type": "gbdt",
            "learning_rate": 0.025, "num_leaves": 31, "max_depth": 5, "min_data_in_leaf": 300,
            "feature_fraction": 0.80, "bagging_fraction": 0.80, "bagging_freq": 1,
            "lambda_l1": 0.5, "lambda_l2": 2.0, "min_gain_to_split": 0.00001, "verbose": -1,
        },
    },
]

WEIGHT_SPECS = [
    {"weight_name": "flat", "note": "all training rows weight=1"},
    {"weight_name": "top_tail_v1", "note": "within-month top20% + abs alpha tail focus, with mild bottom-tail contrast"},
]

RUN_SPECS = []
for p in PARAM_SPECS:
    for w in WEIGHT_SPECS:
        RUN_SPECS.append({
            "run_name": p["param_name"] + "__" + w["weight_name"],
            "param_name": p["param_name"],
            "weight_name": w["weight_name"],
            "fixed_iter": int(p["fixed_iter"]),
            "params": dict(p["params"]),
            "weight_note": w["note"],
        })

print("runs:", len(RUN_SPECS))
for s in RUN_SPECS:
    print(s["run_name"], "iter", s["fixed_iter"])


In [ ]:
def unique_keep_order(cols):
    seen, out = set(), []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": a, "b": b}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            v = corr.iloc[i, j]
            if not pd.isnull(v) and abs(v) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]
    visited, comps = set(), []
    def dfs(x, comp):
        visited.add(x)
        comp.append(x)
        for y in graph[x]:
            if y not in visited:
                dfs(y, comp)
    for col in feature_cols:
        if col not in visited:
            comp = []
            dfs(col, comp)
            comps.append(comp)
    return comps


def select_features_train_only(train_df, candidate_cols):
    cols = unique_keep_order([c for c in candidate_cols if c in train_df.columns])
    missing = train_df[cols].isnull().sum().to_dict()
    keep, remove = [], []
    for comp in build_corr_components(train_df, cols, CORR_THRESHOLD):
        if len(comp) == 1:
            keep.append(comp[0])
        else:
            comp = sorted(comp, key=lambda x: (missing[x], x))
            keep.append(comp[0])
            remove.extend(comp[1:])
    return keep, remove


def split_diag_valid(train_df):
    months = sorted(pd.to_datetime(train_df["rebalance_date"].dropna().unique()))
    n_valid = max(DIAG_VALID_MIN_MONTHS, int(round(len(months) * DIAG_VALID_FRAC)))
    valid_months = set(months[-min(n_valid, max(1, len(months) - 1)):])
    fit = train_df[~train_df["rebalance_date"].isin(valid_months)].copy()
    valid = train_df[train_df["rebalance_date"].isin(valid_months)].copy()
    if fit.empty or valid.empty:
        fit, valid = train_df.copy(), train_df.copy()
    return fit, valid


def prepare_xy(df, feature_cols, target_col, fill_values=None):
    d = df.dropna(subset=[target_col]).copy()
    X = d[feature_cols].replace([np.inf, -np.inf], np.nan)
    y = d[target_col].astype(float)
    if fill_values is None:
        fill_values = X.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.fillna(fill_values).fillna(0)
    return X, y, fill_values, d.index


def calc_sample_weight(df, target_col, weight_name):
    if weight_name == "flat":
        return pd.Series(1.0, index=df.index)
    if weight_name != "top_tail_v1":
        raise ValueError("unknown weight_name: " + str(weight_name))

    g = df.groupby("rebalance_date")[target_col]
    rank_pct = g.rank(pct=True, method="first")
    abs_alpha = df[target_col].abs()
    abs_q80 = abs_alpha.groupby(df["rebalance_date"]).transform(lambda s: s.quantile(0.80))

    w = pd.Series(1.0, index=df.index)
    w += (rank_pct >= 0.80).astype(float) * 1.0
    w += (rank_pct <= 0.20).astype(float) * 0.5
    w += (abs_alpha >= abs_q80).astype(float) * 0.5
    return w.clip(lower=1.0, upper=3.0).astype(float)


In [ ]:
def load_train_df(path):
    if not os.path.exists(path):
        raise IOError("DATA_PATH not found: " + path)
    df = pd.read_csv(path)
    if "code" in df.columns and "stock" not in df.columns:
        df = df.rename(columns={"code": "stock"})
    for col in ["rebalance_date", "feature_date", "next_date"]:
        df[col] = pd.to_datetime(df[col]).dt.normalize()
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=["stock", "rebalance_date", "next_date", TARGET_COL]).copy()

    train = df[(df["rebalance_date"] >= pd.Timestamp(TRAIN_START)) & (df["rebalance_date"] <= pd.Timestamp(TRAIN_END))].copy()
    if REQUIRE_LABEL_END_WITHIN_TRAIN:
        train = train[train["next_date"] <= pd.Timestamp(LABEL_END)].copy()
    return df, train


df_all, train_df = load_train_df(DATA_PATH)
diag_fit_df, diag_valid_df = split_diag_valid(train_df)
feature_cols, removed_cols = select_features_train_only(diag_fit_df, CANDIDATE_COLS)

print("all:", df_all.shape, df_all["rebalance_date"].min(), df_all["rebalance_date"].max())
print("train:", train_df.shape, train_df["rebalance_date"].min(), train_df["rebalance_date"].max(), "months", train_df["rebalance_date"].nunique())
print("diag_fit:", diag_fit_df.shape, "diag_valid:", diag_valid_df.shape)
print("features:", len(feature_cols), "removed:", len(removed_cols))
print(feature_cols)
print(train_df[[TARGET_COL]].describe())


In [ ]:
def train_one_run(run_spec):
    params = dict(run_spec["params"])
    params["seed"] = SEED

    X_train, y_train, fill_values, train_idx = prepare_xy(train_df, feature_cols, TARGET_COL)
    train_base = train_df.loc[train_idx]
    sample_weight = calc_sample_weight(train_base, TARGET_COL, run_spec["weight_name"]).reindex(train_idx).fillna(1.0).values

    model = lgb.train(
        params,
        lgb.Dataset(X_train, label=y_train, weight=sample_weight),
        num_boost_round=max(1, int(run_spec["fixed_iter"])),
        valid_sets=[lgb.Dataset(X_train, label=y_train, weight=sample_weight)],
        valid_names=["train"],
        verbose_eval=False,
    )

    train_pred = np.asarray(model.predict(X_train[feature_cols], num_iteration=run_spec["fixed_iter"])).reshape(-1)
    train_rank_ic = safe_rank_ic(y_train, train_pred)

    X_valid, y_valid, _, valid_idx = prepare_xy(diag_valid_df, feature_cols, TARGET_COL, fill_values)
    valid_pred = np.asarray(model.predict(X_valid[feature_cols], num_iteration=run_spec["fixed_iter"])).reshape(-1)
    diag_rank_ic = safe_rank_ic(y_valid, valid_pred)

    return {
        "model": model,
        "fill_values": fill_values,
        "train_rows": int(len(X_train)),
        "weight_mean": float(np.mean(sample_weight)),
        "weight_max": float(np.max(sample_weight)),
        "train_rank_ic": train_rank_ic,
        "diag_rank_ic": diag_rank_ic,
    }


TRAINED = {}
for spec in RUN_SPECS:
    print("training:", spec["run_name"])
    TRAINED[spec["run_name"]] = train_one_run(spec)
    row = TRAINED[spec["run_name"]]
    print("  train_rank_ic:", row["train_rank_ic"], "diag_rank_ic:", row["diag_rank_ic"], "w_mean:", row["weight_mean"])
    gc.collect()


In [ ]:
export_rows = []

for spec in RUN_SPECS:
    trained = TRAINED[spec["run_name"]]
    research_version = "candidate_v47_{}_{}_alpha1m_{}_{}".format(
        spec["param_name"], spec["weight_name"], TRAIN_START.replace("-", ""), TRAIN_END.replace("-", "")
    )
    model_file = "model_{}.pkl".format(research_version)

    bundle = {
        "objective": "v210_refit_fixed_iter_overlay",
        "research_version": research_version,
        "benchmark": BENCHMARK,
        "train_start": TRAIN_START,
        "train_end": TRAIN_END,
        "label_end": LABEL_END,
        "require_label_end_within_train": bool(REQUIRE_LABEL_END_WITHIN_TRAIN),
        "target_col": TARGET_COL,
        "target_note": "V47 anchor enhancement: alpha_1m label, fixed iter, param/sample-weight family",
        "data_file": DATA_PATH,
        "protocol": "v47_anchor_enhancement_fixed_iter_full_train",
        "training_policy": "expanding",
        "param_set": spec["param_name"],
        "sample_weight_policy": spec["weight_name"],
        "sample_weight_note": spec["weight_note"],
        "final_role": "v47_anchor_enhancement_candidate",
        "base_params": spec["params"],
        "base_model": trained["model"],
        "base_feature_cols": list(feature_cols),
        "base_fill_values": dict(trained["fill_values"]),
        "base_best_iter": int(spec["fixed_iter"]),
        "es_best_iter": np.nan,
        "model_iter": int(spec["fixed_iter"]),
        "fixed_iter": int(spec["fixed_iter"]),
        "base_inner_metrics": {
            "train_rank_ic": float(trained["train_rank_ic"]) if not pd.isnull(trained["train_rank_ic"]) else np.nan,
            "diag_rank_ic": float(trained["diag_rank_ic"]) if not pd.isnull(trained["diag_rank_ic"]) else np.nan,
        },
        "base_removed_features": list(removed_cols),
        "residual_model": None,
        "residual_feature_cols": [],
        "residual_fill_values": {},
        "overlay_weight": 0.0,
        "overlay_mode": "direct",
        "top_n_candidates": TOP_N_CANDIDATES,
        "stock_num": STOCK_NUM,
        "industry_cap_ratio": INDUSTRY_CAP_RATIO,
        "requires_v4_feature_adapter": True,
        "uses_time_weight": False,
        "uses_sample_weight": spec["weight_name"] != "flat",
        "uses_current_valid_for_training": False,
    }

    out_path = os.path.join(OUT_DIR, model_file)
    with open(out_path, "wb") as f:
        pickle.dump(bundle, f, protocol=2)

    export_rows.append({
        "run_name": spec["run_name"],
        "param_name": spec["param_name"],
        "weight_name": spec["weight_name"],
        "model_file": model_file,
        "model_path": out_path,
        "train_start": TRAIN_START,
        "train_end": TRAIN_END,
        "target_col": TARGET_COL,
        "fixed_iter": int(spec["fixed_iter"]),
        "train_rank_ic": trained["train_rank_ic"],
        "diag_rank_ic": trained["diag_rank_ic"],
        "train_rows": trained["train_rows"],
        "weight_mean": trained["weight_mean"],
        "weight_max": trained["weight_max"],
        "feature_count": len(feature_cols),
        "removed_feature_count": len(removed_cols),
    })

export_manifest_df = pd.DataFrame(export_rows).sort_values(["param_name", "weight_name"])
manifest_path = os.path.join(OUT_DIR, "v47_anchor_enhancement_export_manifest.csv")
export_manifest_df.to_csv(manifest_path, index=False)
print(export_manifest_df)
print("manifest:", manifest_path)
print("saved files:")
for row in export_rows:
    print("  " + row["model_path"])


In [ ]:
required = [
    "objective", "base_model", "base_feature_cols", "base_fill_values",
    "residual_feature_cols", "residual_fill_values", "overlay_weight", "overlay_mode",
]
for row in export_rows:
    loaded = pickle.load(open(row["model_path"], "rb"))
    missing = [k for k in required if k not in loaded]
    print(row["run_name"], "missing keys:", missing)
    print("  objective:", loaded["objective"], "mode:", loaded["overlay_mode"], "features:", len(loaded["base_feature_cols"]), "iter:", loaded["fixed_iter"])


## 结论填写区

回测后填写：

- 保留：
- 废弃：
- 是否 sample weight 有帮助：
- 是否更强正则有帮助：
- 下一步：
